<a href="https://colab.research.google.com/github/Sarah-0405/Cold_Spots_Bayern/blob/main/Gi_Analyse_Puffergebiete_test.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

input Daten:
- KNN-Matrizen
- LST-Daten in den Puffergebieten

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
!pip install esda pysal

  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.6/56.6 kB 4.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 142.0/142.0 kB 6.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.3/61.3 kB 5.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 882.2/882.2 kB 24.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.9/47.9 kB 4.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 57.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.5/63.5 kB 6.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 141.6/141.6 kB 12.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 53.9/53.9 kB 4.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.4/41.4 kB 3.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 248.1/248.1 kB 19.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 389.9/389.9 kB 21.0 MB/s eta 0:00:00
   ━━━━━

In [ ]:
import geopandas as gpd
import pandas as pd
from esda.getisord import G_Local
import numpy as np
import os
import pysal as ps
import libpysal.io as ps_io


#LST-Daten laden

In [ ]:
# 1. LST-Werte laden aus Ordner Puffergebiete (non aggregated) (oder aus lokalem Ordner)

# Define the path to the "Puffergebiete" folder in Google Drive
input_path = "/content/drive/MyDrive/Cold Spots Bayern/Puffergebiete" # oder zb input_path = "input_lst_daten" mit nur 1 oder 2 Städten zum Test

# Initialize dictionaries to store file paths categorized by city and time period
gpkg_files_2024 = {}
gpkg_files_2019_2024 = {}

# Check if the folder exists
if os.path.exists(input_path):
    # List all files in the folder
    for filename in os.listdir(input_path):
        # Check if the file is a GeoPackage file and starts with "joined_"
        if filename.endswith(".gpkg") and filename.startswith("joined_"):
            file_path = os.path.join(input_path, filename)

            # Determine time period based on the filename
            time_period = None
            if "2019_2024" in filename:
                time_period = "2019_2024"
            elif "2024" in filename and "2019_2024" not in filename: # Check for 2024, but not part of 2019_2024
                time_period = "2024"
            # If neither 2024 nor 2019_2024 are explicitly in the name, this file might not fit the expected pattern.
            # We will only process files with a clear time period identified.


            # Extract city name - Assuming city name is after "mit_lst_" and before the year information
            city_name = None
            if time_period: # Only try to extract city name if a time period was determined
                try:
                    # Find the part after "mit_lst_"
                    mit_lst_part = filename.split("mit_lst_")[-1]

                    # Remove the time period and .gpkg extension to get the city name part
                    city_name = mit_lst_part.replace(f"{time_period}.gpkg", "").replace(".gpkg", "").strip("_")


                    # Handle umlauts and special characters in city names if necessary
                    # Example: city_name = city_name.replace("ü", "ue").replace("ä", "ae").replace("ö", "oe")


                except Exception as e:
                    print(f"Warning: Could not parse city name from filename: {filename} - {e}. Skipping.")
                    city_name = None # Ensure city_name is None if parsing fails


            if city_name and time_period:
                 if time_period == "2024":
                     gpkg_files_2024[city_name] = file_path
                 elif time_period == "2019_2024":
                     gpkg_files_2019_2024[city_name] = file_path
                 # No else needed here, as we only proceed if time_period was identified


    # Print the categorized files
    print("GeoPackage files for 2024 (joined):")
    for city, path in gpkg_files_2024.items():
        print(f"  {city}: {path}")

    print("\nGeoPackage files for 2019-2024 (joined):")
    for city, path in gpkg_files_2019_2024.items():
        print(f"  {city}: {path}")

else:
    print(f"The folder '{input_path}' was not found. Please check the path.")

GeoPackage files for 2024 (joined):
  2024_Aschaffenburg: /content/drive/MyDrive/Cold Spots Bayern/Puffergebiete/joined_350mpuffer_um_Gitterzelle_mit_lst_2024_Aschaffenburg.gpkg
  2024_Augsburg: /content/drive/MyDrive/Cold Spots Bayern/Puffergebiete/joined_350mpuffer_um_Gitterzelle_mit_lst_2024_Augsburg.gpkg
  2024_Bamberg: /content/drive/MyDrive/Cold Spots Bayern/Puffergebiete/joined_350mpuffer_um_Gitterzelle_mit_lst_2024_Bamberg.gpkg
  2024_Bayreuth: /content/drive/MyDrive/Cold Spots Bayern/Puffergebiete/joined_350mpuffer_um_Gitterzelle_mit_lst_2024_Bayreuth.gpkg
  2024_Erlangen: /content/drive/MyDrive/Cold Spots Bayern/Puffergebiete/joined_350mpuffer_um_Gitterzelle_mit_lst_2024_Erlangen.gpkg
  2024_Fürth: /content/drive/MyDrive/Cold Spots Bayern/Puffergebiete/joined_350mpuffer_um_Gitterzelle_mit_lst_2024_Fürth.gpkg
  2024_Ingolstadt: /content/drive/MyDrive/Cold Spots Bayern/Puffergebiete/joined_350mpuffer_um_Gitterzelle_mit_lst_2024_Ingolstadt.gpkg
  2024_Kempten_(Allgäu): /conte

# Gi Analyse 2024

Loop für jeden Zeitraum der durch alle Städte iteriert, erst die KNN Matrix läd und dann Gi berechnet, speichtert und lokalen Speicher dann wieder frei macht

In [ ]:
import geopandas as gpd
import pandas as pd
from esda.getisord import G_Local
import numpy as np
import os
import pysal as ps # Import pysal here for other potential uses
import libpysal.io as ps_io # Import libpysal.io as ps_io


output_dir = "gi_star_results"
os.makedirs(output_dir, exist_ok=True)

def calculate_hotspots(gdf, w):
    gdf = gdf.copy()
    g = G_Local(gdf["avg_summer_LST_Celsius"], w)
    gdf["Gi*"] = g.Zs  # Z-score
    gdf["p_sim"] = g.p_sim # p-value from simulations
    return gdf

# Definieren Sie den Ordner, aus dem die Gewichtsmatrizen geladen werden sollen
input_weights_folder = "/content/drive/MyDrive/Cold Spots Bayern/Pufferzonen_KNN_Weights"

# --- Startstadt definieren (manuell anpassen, falls nötig) ---
# Tragen Sie hier den Namen der Stadt ein, ab der die Verarbeitung beginnen soll.
# Lassen Sie es leer (start_city = ""), um alle Städte zu verarbeiten.
start_city = "" # Beispiel: "2024_Passau" oder "2019_2024_Passau" - muss mit dem Schlüssel im Dictionary gpkg_files_... übereinstimmen

# Definieren Sie die Spalten, die im finalen Output behalten werden sollen
# Fügen Sie hier weitere Spalten hinzu, falls benötigt (z.B. eine ID-Spalte, falls relevant)
columns_to_keep = ['latitude', 'longitude', 'avg_summer_LST_Celsius', 'geometry', 'Gi*', 'p_sim']


# --- Verarbeitung für Zeitraum 2024 ---
print("\n--- Verarbeitung für Zeitraum 2024 ---")
gpkg_files_to_process_2024 = gpkg_files_2024
cities_to_process_2024 = list(gpkg_files_to_process_2024.keys())

# Anwenden der Startstadt, falls definiert
if start_city:
    try:
        start_index_2024 = cities_to_process_2024.index(start_city)
        cities_to_process_2024 = cities_to_process_2024[start_index_2024:]
        print(f"\nStarte Berechnung für 2024 ab: {start_city}")
    except ValueError:
        print(f"Warnung: Start city '{start_city}' not found in the list of cities for 2024. Processing all cities for 2024.")


print("\nStädte, die verarbeitet werden (2024):", cities_to_process_2024)


for city_key in cities_to_process_2024:
    print(f"\nBearbeite {city_key} (2024)...")

    # Extrahiere den reinen Stadtnamen ohne Jahrespräfix aus dem Dictionary-Schlüssel
    # Annahme: Schlüssel ist im Format "Jahr_Stadt" oder "Jahr_Jahr_Stadt"
    city_parts = city_key.split("_")
    # Finde den ersten Teil, der kein Jahr ist (oder nimm alle Teile nach dem ersten Jahr)
    pure_city_name_parts = [p for p in city_parts if not (p.isdigit() and len(p) == 4) and p != "2019_2024"]
    pure_city_name = "_".join(pure_city_name_parts)

    # Handling von Umlauten und Sonderzeichen im reinen Stadtnamen für den Dateinamen der Gewichte
    # Passen Sie die Bereinigung an, um spezifische Fälle wie Kempten_(Allgäu) zu behandeln
    pure_city_name_cleaned = pure_city_name.replace(' ', '_').replace('-', '_')
    # Spezifische Behandlung für Kempten_(Allgäu) basierend auf Dateinamen
    if "Kempten_(Allgäu)" in city_key:
         pure_city_name_cleaned = "Kempten_(Allgäu)"
    else:
        # Allgemeine Bereinigung für andere Städte
        pure_city_name_cleaned = pure_city_name_cleaned.replace('ä', 'ae').replace('ö', 'oe').replace('ü', 'ue').replace('ß', 'ss')
        pure_city_name_cleaned = pure_city_name_cleaned.replace('(', '').replace(')', '') # Entferne Klammern


    # Konstruiere den erwarteten Dateinamen der Gewichtsmatrix basierend auf dem bereinigten reinen Stadtnamen
    # FÜR 2024 WIRD KEINE JAHRESZAHL IM GEWICHTS-DATEINAMEN ERWARTET
    weight_file_name = f"knn_weights_{pure_city_name_cleaned}.gal"
    weight_file_path = os.path.join(input_weights_folder, weight_file_name)


    # Pfad zur GeoPackage-Datei für die aktuelle Stadt und den aktuellen Zeitraum (verwende den originalen Schlüssel)
    gpkg_file_path = gpkg_files_to_process_2024[city_key]


    # Überprüfen, ob die GeoPackage-Datei existiert
    if not os.path.exists(gpkg_file_path):
        print(f"Warnung: GeoPackage-Datei nicht gefunden für {city_key}: {gpkg_file_path} — überspringe.")
        continue

    # Überprüfen, ob die Gewichtsdatei existiert
    if not os.path.exists(weight_file_path):
        print(f"Warnung: Gewichtsdatei nicht gefunden für {city_key} (erwartet: {weight_file_name}): {weight_file_path} — überspringe.")
        continue


    try:
        # Laden Sie das GeoDataFrame für die aktuelle Stadt
        print(f"  Lade GeoPackage für {city_key} (2024)...")
        gdf = gpd.read_file(gpkg_file_path)
        print(f"  GeoPackage für {city_key} (2024) erfolgreich geladen ({len(gdf)} Features).")

        # Laden Sie die Gewichtsmatrix für die aktuelle Stadt
        print(f"  Lade Gewichtsmatrix für {pure_city_name}...")
        w = ps_io.open(weight_file_path, mode='r').read()
        print(f"  Gewichtsmatrix für {pure_city_name} erfolgreich geladen.")

        # Print, indicating the start of Gi* analysis
        print(f"  Starte Gi* Analyse für {city_key} (2024)...")

        # Ensure the GeoDataFrame has the correct index for the weights
        if not gdf.index.equals(pd.RangeIndex(len(gdf))):
             gdf = gdf.reset_index(drop=True)
             print(f"  Zurücksetzen des Index für {city_key} (2024).")

        # Ensure the weight matrix has the same number of observations as the geodataframe
        if w.n != len(gdf):
            print(f"Warnung: Anzahl der Beobachtungen in Gewichten ({w.n}) und GeoDataFrame ({len(gdf)}) für {city_key} (2024) stimmen nicht überein — überspringe.")
            del gdf # Speicher freigeben
            del w # Speicher freigeben
            continue

        # Gi* berechnen
        if "avg_summer_LST_Celsius" not in gdf.columns:
             print(f"Warnung: Spalte 'avg_summer_LST_Celsius' nicht gefunden in GeoDataFrame für {city_key} (2024) — überspringe.")
             del gdf
             del w
             continue

        result_gdf = calculate_hotspots(gdf, w)

        # --- Spalten auswählen und bereinigen ---
        # Stellen Sie sicher, dass alle gewünschten Spalten vorhanden sind, bevor Sie auswählen
        final_columns = [col for col in columns_to_keep if col in result_gdf.columns]
        result_gdf = result_gdf[final_columns]
        print(f"  Daten bereinigt: Behalte nur Spalten {final_columns}")
        # --- Ende Spaltenauswahl ---


        # Speichern Sie das Ergebnis als GeoPackage mit Jahres-Endung
        output_subdir_2024 = os.path.join(output_dir, "2024") # Speichern in Unterordner pro Zeitraum
        os.makedirs(output_subdir_2024, exist_ok=True)
        out_path = os.path.join(output_subdir_2024, f"{pure_city_name.replace(' ', '_')}_2024.gpkg") # Füge Jahres-Endung hinzu, verwende reinen Stadtnamen

        result_gdf.to_file(out_path, driver="GPKG")
        print(f"Ergebnis für {city_key} (2024) gespeichert: {out_path}")

        # Speicher freigeben
        del result_gdf
        del gdf # Speicher für das GeoDataFrame freigeben
        del w # Speicher für die Gewichtsmatrix freigeben


    except Exception as e:
        print(f"Fehler bei der Bearbeitung von {city_key} (2024): {e}")
        if 'gdf' in locals() and gdf is not None:
             del gdf
        if 'w' in locals() and w is not None:
             del w


print("\nGi* Analyse für 2024 abgeschlossen.")

# --- Verarbeitung für Zeitraum 2019-2024 ---
print("\n--- Verarbeitung für Zeitraum 2019-2024 ---")
gpkg_files_to_process_2019_2024 = gpkg_files_2019_2024
cities_to_process_2019_2024 = list(gpkg_files_to_process_2019_2024.keys())

# Anwenden der Startstadt, falls definiert
if start_city:
    try:
        start_index_2019_2024 = cities_to_process_2019_2024.index(start_city)
        cities_to_process_2019_2024 = cities_to_process_2019_2024[start_index_2019_2024:]
        print(f"Starte Berechnung für 2019-2024 ab: {start_city}")
    except ValueError:
        # Wenn die Startstadt nicht im 2019-2024 Datensatz gefunden wird,
        # bedeutet das, dass sie entweder bereits verarbeitet wurde
        # oder in diesem Zeitraum nicht existiert.
        # Wir verarbeiten dann einfach alle verbleibenden Städte in dieser Liste.
        print(f"Warnung: Start city '{start_city}' not found in the list of cities for 2019-2024 (possibly already processed or not in this period). Processing remaining cities for 2019-2024.")


print("\nStädte, die verarbeitet werden (2019-2024):", cities_to_process_2019_2024)

for city_key in cities_to_process_2019_2024:
    print(f"\nBearbeite {city_key} (2019-2024)...")

    # Extrahiere den reinen Stadtnamen ohne Jahrespräfix aus dem Dictionary-Schlüssel
    # Annahme: Schlüssel ist im Format "Jahr_Jahr_Stadt" oder "Jahr_Stadt"
    city_parts = city_key.split("_")
    # Finde den ersten Teil, der kein Jahr ist (oder nimm alle Teile nach dem ersten Jahr)
    pure_city_name_parts = [p for p in city_parts if not (p.isdigit() and len(p) == 4) and p != "2019_2024"]
    pure_city_name = "_".join(pure_city_name_parts)

    # Handling von Umlauten und Sonderzeichen im reinen Stadtnamen für den Dateinamen der Gewichte
    # Passen Sie die Bereinigung an, um spezifische Fälle wie Kempten_(Allgäu) zu behandeln
    pure_city_name_cleaned = pure_city_name.replace(' ', '_').replace('-', '_')
    # Spezifische Behandlung für Kempten_(Allgäu) basierend auf Dateinamen
    if "Kempten_(Allgäu)" in city_key:
         pure_city_name_cleaned = "Kempten_(Allgäu)"
    else:
        # Allgemeine Bereinigung für andere Städte
        pure_city_name_cleaned = pure_city_name_cleaned.replace('ä', 'ae').replace('ö', 'oe').replace('ü', 'ue').replace('ß', 'ss')
        pure_city_name_cleaned = pure_city_name_cleaned.replace('(', '').replace(')', '') # Entferne Klammern


    # Konstruiere den erwarteten Dateinamen der Gewichtsmatrix basierend auf dem bereinigten reinen Stadtnamen
    # FÜR 2019-2024 WIRD DIE JAHRESZAHL IM GEWICHTS-DATEINAMEN ERWARTET
    weight_file_name = f"knn_weights_2019_2024_{pure_city_name_cleaned}.gal" # Hinzufügen der Jahresangabe
    weight_file_path = os.path.join(input_weights_folder, weight_file_name)

    # Pfad zur GeoPackage-Datei für die aktuelle Stadt und den aktuellen Zeitraum (verwende den originalen Schlüssel)
    gpkg_file_path = gpkg_files_to_process_2019_2024[city_key]

    # Überprüfen, ob die GeoPackage-Datei existiert
    if not os.path.exists(gpkg_file_path):
        print(f"Warnung: GeoPackage-Datei nicht gefunden für {city_key}: {gpkg_file_path} — überspringe.")
        continue

    # Überprüfen, ob die Gewichtsdatei existiert
    if not os.path.exists(weight_file_path):
        print(f"Warnung: Gewichtsdatei nicht gefunden für {city_key} (erwartet: {weight_file_name}): {weight_file_path} — überspringe.")
        continue

    try:
        # Laden Sie das GeoDataFrame für die aktuelle Stadt
        print(f"  Lade GeoPackage für {city_key} (2019-2024)...")
        gdf = gpd.read_file(gpkg_file_path)
        print(f"  GeoPackage für {city_key} (2019-2024) erfolgreich geladen ({len(gdf)} Features).")


        # Laden Sie die Gewichtsmatrix für die aktuelle Stadt
        print(f"  Lade Gewichtsmatrix für {pure_city_name}...")
        w = ps_io.open(weight_file_path, mode='r').read()
        print(f"  Gewichtsmatrix für {pure_city_name} erfolgreich geladen ({w.n} Beobachtungen).")

        # Print, indicating the start of Gi* analysis
        print(f"  Starte Gi* Analyse für {city_key} (2019-2024)...")

        # Ensure the GeoDataFrame has the correct index for the weights
        if not gdf.index.equals(pd.RangeIndex(len(gdf))):
             gdf = gdf.reset_index(drop=True)
             print(f"  Zurücksetzen des Index für {city_key} (2019-2024).")

        # Ensure the weight matrix has the same number of observations as the geodataframe
        if w.n != len(gdf):
            print(f"Warnung: Anzahl der Beobachtungen in Gewichten ({w.n}) und GeoDataFrame ({len(gdf)}) für {city_key} (2019-2024) stimmen nicht überein — überspringe.")
            del gdf # Speicher freigeben
            del w # Speicher freigeben
            continue

        # Gi* berechnen
        if "avg_summer_LST_Celsius" not in gdf.columns:
             print(f"Warnung: Spalte 'avg_summer_LST_Celsius' nicht gefunden in GeoDataFrame für {city_key} (2019-2024) — überspringe.")
             del gdf
             del w
             continue

        result_gdf = calculate_hotspots(gdf, w)

        # --- Spalten auswählen und bereinigen ---
        # Stellen Sie sicher, dass alle gewünschten Spalten vorhanden sind, bevor Sie auswählen
        final_columns = [col for col in columns_to_keep if col in result_gdf.columns]
        result_gdf = result_gdf[final_columns]
        print(f"  Daten bereinigt: Behalte nur Spalten {final_columns}")
        # --- Ende Spaltenauswahl ---


        # Speichern Sie das Ergebnis als GeoPackage mit Jahres-Endung
        output_subdir_2019_2024 = os.path.join(output_dir, "2019-2024")
        os.makedirs(output_subdir_2019_2024, exist_ok=True)
        out_path = os.path.join(output_subdir_2019_2024, f"{pure_city_name.replace(' ', '_')}_2019_2024.gpkg") # Füge Jahres-Endung hinzu, verwende reinen Stadtnamen


        result_gdf.to_file(out_path, driver="GPKG")
        print(f"Ergebnis für {city_key} (2019-2024) gespeichert: {out_path}")

        # Speicher freigeben
        del result_gdf
        del gdf # Speicher für das GeoDataFrame freigeben
        del w # Speicher für die Gewichtsmatrix freigeben


    except Exception as e:
        print(f"Fehler bei der Bearbeitung von {city_key} (2019-2024): {e}")
        if 'gdf' in locals() and gdf is not None:
             del gdf
        if 'w' in locals() and w is not None:
             del w


print("\nGi* Analyse für 2019-2024 abgeschlossen.")
print("\nGesamte Gi* Analyse abgeschlossen.")


--- Verarbeitung für Zeitraum 2024 ---

Städte, die verarbeitet werden (2024): ['2024_Passau', '2024_Nuremberg', '2024_Munich', '2024_Landshut', '2024_Ingolstadt', '2024_Erlangen', '2024_Bayreuth', '2024_Schweinfurt', '2024_Rosenheim', '2024_Bamberg', '2024_Augsburg', '2024_Kempten_(Allgäu)', '2024_Aschaffenburg', '2024_Fürth', '2024_Würzburg', '2024_Regensburg']

Bearbeite 2024_Passau (2024)...
  Lade GeoPackage für 2024_Passau (2024)...
  GeoPackage für 2024_Passau (2024) erfolgreich geladen (786919 Features).
  Lade Gewichtsmatrix für Passau...


/usr/local/lib/python3.12/dist-packages/libpysal/io/iohandlers/gal.py:185: UserWarning: The weights matrix is not fully connected: 
 There are 33895 disconnected components.
  w = W(neighbors, id_order=ids)


  Gewichtsmatrix für Passau erfolgreich geladen.
  Starte Gi* Analyse für 2024_Passau (2024)...


KeyboardInterrupt: 

Anzahl der Beobachtungen der KNN-Elemente in beiden Datensätzen überprüfen (Passau)

In [ ]:
# Pfade zu den Dateien für Passau
input_weights_folder = "/content/drive/MyDrive/Cold Spots Bayern/Pufferzonen_KNN_Weights"
input_data_folder = "/content/drive/MyDrive/Cold Spots Bayern/Puffergebiete" # Dieser Ordner ist bereits in oQopEVffPzMZ definiert, zur Klarheit hier wiederholt

# Dateinamen für Passau
gpkg_file_name_passau_2024 = "joined_350mpuffer_um_Gitterzelle_mit_lst_2024_Passau.gpkg"
gpkg_file_name_passau_2019_2024 = "joined_350mpuffer_um_Gitterzelle_mit_lst_2019_2024_Passau.gpkg"
weight_file_name_passau_2024 = "knn_weights_Passau.gal" # Dateiname der 2024er Gewichte ohne Jahreszahl
weight_file_name_passau_2019_2024 = "knn_weights_2019_2024_Passau.gal" # Dateiname der 2019-2024er Gewichte

# Vollständige Pfade
gpkg_file_path_passau_2024 = os.path.join(input_data_folder, gpkg_file_name_passau_2024)
gpkg_file_path_passau_2019_2024 = os.path.join(input_data_folder, gpkg_file_name_passau_2019_2024)
weight_file_path_passau_2024 = os.path.join(input_weights_folder, weight_file_name_passau_2024)
weight_file_path_passau_2019_2024 = os.path.join(input_weights_folder, weight_file_name_passau_2019_2024)


print("--- Anzahl der Beobachtungen für Passau ---")

# 2024 Daten und Gewichte
print("\nZeitraum: 2024")
if os.path.exists(gpkg_file_path_passau_2024):
    try:
        gdf_passau_2024 = gpd.read_file(gpkg_file_path_passau_2024)
        print(f"  GeoPackage '{gpkg_file_name_passau_2024}': {len(gdf_passau_2024)} Features")
        del gdf_passau_2024 # Speicher freigeben
    except Exception as e:
        print(f"  Fehler beim Laden des GeoPackage 2024: {e}")
else:
    print(f"  GeoPackage '{gpkg_file_name_passau_2024}' nicht gefunden.")


if os.path.exists(weight_file_path_passau_2024):
    try:
        w_passau_2024 = ps_io.open(weight_file_path_passau_2024, mode='r').read()
        print(f"  Gewichtsmatrix '{weight_file_name_passau_2024}': {w_passau_2024.n} Beobachtungen")
        del w_passau_2024 # Speicher freigeben
    except Exception as e:
        print(f"  Fehler beim Laden der Gewichtsmatrix 2024: {e}")
else:
    print(f"  Gewichtsmatrix '{weight_file_name_passau_2024}' nicht gefunden.")


# 2019-2024 Daten und Gewichte
print("\nZeitraum: 2019-2024")
if os.path.exists(gpkg_file_path_passau_2019_2024):
    try:
        gdf_passau_2019_2024 = gpd.read_file(gpkg_file_path_passau_2019_2024)
        print(f"  GeoPackage '{gpkg_file_name_passau_2019_2024}': {len(gdf_passau_2019_2024)} Features")
        del gdf_passau_2019_2024 # Speicher freigeben
    except Exception as e:
        print(f"  Fehler beim Laden des GeoPackage 2019-2024: {e}")
else:
    print(f"  GeoPackage '{gpkg_file_name_passau_2019_2024}' nicht gefunden.")


if os.path.exists(weight_file_path_passau_2019_2024):
    try:
        w_passau_2019_2024 = ps_io.open(weight_file_path_passau_2019_2024, mode='r').read()
        print(f"  Gewichtsmatrix '{weight_file_name_passau_2019_2024}': {w_passau_2019_2024.n} Beobachtungen")
        del w_passau_2019_2024 # Speicher freigeben
    except Exception as e:
        print(f"  Fehler beim Laden der Gewichtsmatrix 2019-2024: {e}")
else:
    print(f"  Gewichtsmatrix '{weight_file_name_passau_2019_2024}' nicht gefunden.")

print("\n--- Ende der Übersicht ---")

--- Anzahl der Beobachtungen für Passau ---

Zeitraum: 2024
  GeoPackage 'joined_350mpuffer_um_Gitterzelle_mit_lst_2024_Passau.gpkg': 786919 Features


/usr/local/lib/python3.12/dist-packages/libpysal/io/iohandlers/gal.py:185: UserWarning: The weights matrix is not fully connected: 
 There are 33895 disconnected components.
  w = W(neighbors, id_order=ids)


  Gewichtsmatrix 'knn_weights_Passau.gal': 786919 Beobachtungen

Zeitraum: 2019-2024
  GeoPackage 'joined_350mpuffer_um_Gitterzelle_mit_lst_2019_2024_Passau.gpkg': 799228 Features


/usr/local/lib/python3.12/dist-packages/libpysal/io/iohandlers/gal.py:185: UserWarning: The weights matrix is not fully connected: 
 There are 34322 disconnected components.
  w = W(neighbors, id_order=ids)


  Gewichtsmatrix 'knn_weights_2019_2024_Passau.gal': 799228 Beobachtungen

--- Ende der Übersicht ---


bestätigt unterschiedliche Anzahl der Beobachtungen für 2019-2024 und 2024 => eigene KNN Matrizen pro Jahr benötigt und damit dann Gi*

#Gi Analyse 2019-2024 mit angepassten KNN


In [ ]:
import geopandas as gpd
import pandas as pd
from esda.getisord import G_Local
import numpy as np
import os
import pysal as ps
import libpysal.io as ps_io


output_dir = "gi_star_results"
os.makedirs(output_dir, exist_ok=True)

def calculate_hotspots(gdf, w):
    gdf = gdf.copy()
    g = G_Local(gdf["avg_summer_LST_Celsius"], w)
    gdf["Gi*"] = g.Zs  # Z-score
    gdf["p_sim"] = g.p_sim # p-value from simulations
    return gdf

# Definieren Sie die Spalten, die im finalen Output behalten werden sollen
# Fügen Sie hier weitere Spalten hinzu, falls benötigt (z.B. eine ID-Spalte, falls relevant)
columns_to_keep = ['latitude', 'longitude', 'avg_summer_LST_Celsius', 'geometry', 'Gi*', 'p_sim']

# Definieren Sie den Ordner, aus dem die Gewichtsmatrizen geladen werden sollen
input_weights_folder = "/content/drive/MyDrive/Cold Spots Bayern/Pufferzonen_KNN_Weights"


# --- Verarbeitung für Zeitraum 2019-2024 ---
print("\n--- Verarbeitung für Zeitraum 2019-2024 ---")
# Stellen Sie sicher, dass gpkg_files_2019_2024 aus einer vorherigen Zelle geladen wurde
if 'gpkg_files_2019_2024' not in locals():
    print("Fehler: Das Dictionary 'gpkg_files_2019_2024' wurde nicht gefunden. Bitte stellen Sie sicher, dass die Zelle zum Laden der Daten ('oQopEVffPzMZ') ausgeführt wurde.")
else:
    gpkg_files_to_process_2019_2024 = gpkg_files_2019_2024
    cities_to_process_2019_2024 = list(gpkg_files_to_process_2019_2024.keys())

    # Optional: Startstadt definieren, falls Sie nicht alle Städte von Anfang an verarbeiten möchten
    start_city = "" # Beispiel: "2019_2024_Passau"
    if start_city:
        try:
            start_index_2019_2024 = cities_to_process_2019_2024.index(start_city)
            cities_to_process_2019_2024 = cities_to_process_2019_2024[start_index_2019_2024:]
            print(f"Starte Berechnung für 2019-2024 ab: {start_city}")
        except ValueError:
            print(f"Warnung: Start city '{start_city}' not found in the list of cities for 2019-2024. Processing all cities for 2019-2024.")


    print("\nStädte, die verarbeitet werden (2019-2024):", cities_to_process_2019_2024)

    for city_key in cities_to_process_2019_2024:
        print(f"\nBearbeite {city_key} (2019-2024)...")

        # Extrahiere den reinen Stadtnamen ohne Jahrespräfix aus dem Dictionary-Schlüssel
        # Annahme: Schlüssel ist im Format "Jahr_Jahr_Stadt" oder "Jahr_Stadt"
        city_parts = city_key.split("_")
        # Finde den ersten Teil, der kein Jahr ist (oder nimm alle Teile nach dem ersten Jahr)
        pure_city_name_parts = [p for p in city_parts if not (p.isdigit() and len(p) == 4) and p != "2019_2024"]
        pure_city_name = "_".join(pure_city_name_parts)

        # Handling von Umlauten und Sonderzeichen im reinen Stadtnamen für den Dateinamen der Gewichte
        # Passen Sie die Bereinigung an, um spezifische Fälle wie Kempten_(Allgäu) zu behandeln
        pure_city_name_cleaned = pure_city_name.replace(' ', '_').replace('-', '_')
        # Spezifische Behandlung für Kempten_(Allgäu) basierend auf Dateinamen
        if "Kempten_(Allgäu)" in city_key:
             pure_city_name_cleaned = "Kempten_(Allgäu)"
        else:
            # Allgemeine Bereinigung für andere Städte
            pure_city_name_cleaned = pure_city_name_cleaned.replace('ä', 'ae').replace('ö', 'oe').replace('ü', 'ue').replace('ß', 'ss')
            pure_city_name_cleaned = pure_city_name_cleaned.replace('(', '').replace(')', '') # Entferne Klammern


        # Konstruiere den erwarteten Dateinamen der Gewichtsmatrix basierend auf dem bereinigten reinen Stadtnamen
        # FÜR 2019-2024 WIRD DIE JAHRESZAHL IM GEWICHTS-DATEINAMEN ERWARTET
        weight_file_name = f"knn_weights_2019_2024_{pure_city_name_cleaned}.gal" # Hinzufügen der Jahresangabe
        weight_file_path = os.path.join(input_weights_folder, weight_file_name)

        # Pfad zur GeoPackage-Datei für die aktuelle Stadt und den aktuellen Zeitraum (verwende den originalen Schlüssel)
        gpkg_file_path = gpkg_files_to_process_2019_2024[city_key]

        # Überprüfen, ob die GeoPackage-Datei existiert
        if not os.path.exists(gpkg_file_path):
            print(f"Warnung: GeoPackage-Datei nicht gefunden für {city_key}: {gpkg_file_path} — überspringe.")
            continue

        # Überprüfen, ob die Gewichtsdatei existiert
        if not os.path.exists(weight_file_path):
            print(f"Warnung: Gewichtsdatei nicht gefunden für {city_key} (erwartet: {weight_file_name}): {weight_file_path} — überspringe.")
            continue

        try:
            # Laden Sie das GeoDataFrame für die aktuelle Stadt
            print(f"  Lade GeoPackage für {city_key} (2019-2024)...")
            gdf = gpd.read_file(gpkg_file_path)
            print(f"  GeoPackage für {city_key} (2019-2024) erfolgreich geladen ({len(gdf)} Features).")


            # Laden Sie die Gewichtsmatrix für die aktuelle Stadt
            print(f"  Lade Gewichtsmatrix für {pure_city_name}...")
            w = ps_io.open(weight_file_path, mode='r').read()
            print(f"  Gewichtsmatrix für {pure_city_name} erfolgreich geladen ({w.n} Beobachtungen).")

            # Print, indicating the start of Gi* analysis
            print(f"  Starte Gi* Analyse für {city_key} (2019-2024)...")

            # Ensure the GeoDataFrame has the correct index for the weights
            if not gdf.index.equals(pd.RangeIndex(len(gdf))):
                 gdf = gdf.reset_index(drop=True)
                 print(f"  Zurücksetzen des Index für {city_key} (2019-2024).")

            # Ensure the weight matrix has the same number of observations as the geodataframe
            if w.n != len(gdf):
                print(f"Warnung: Anzahl der Beobachtungen in Gewichten ({w.n}) und GeoDataFrame ({len(gdf)}) für {city_key} (2019-2024) stimmen nicht überein — überspringe.")
                del gdf # Speicher freigeben
                del w # Speicher freigeben
                continue

            # Gi* berechnen
            if "avg_summer_LST_Celsius" not in gdf.columns:
                 print(f"Warnung: Spalte 'avg_summer_LST_Celsius' nicht gefunden in GeoDataFrame für {city_key} (2019-2024) — überspringe.")
                 del gdf
                 del w
                 continue

            result_gdf = calculate_hotspots(gdf, w)

            # --- Spalten auswählen und bereinigen ---
            # Stellen Sie sicher, dass alle gewünschten Spalten vorhanden sind, bevor Sie auswählen
            final_columns = [col for col in columns_to_keep if col in result_gdf.columns]
            result_gdf = result_gdf[final_columns]
            print(f"  Daten bereinigt: Behalte nur Spalten {final_columns}")
            # --- Ende Spaltenauswahl ---


            # Speichern Sie das Ergebnis als GeoPackage mit Jahres-Endung
            output_subdir_2019_2024 = os.path.join(output_dir, "2019-2024")
            os.makedirs(output_subdir_2019_2024, exist_ok=True)
            out_path = os.path.join(output_subdir_2019_2024, f"{pure_city_name.replace(' ', '_')}_2019_2024.gpkg") # Füge Jahres-Endung hinzu, verwende reinen Stadtnamen


            result_gdf.to_file(out_path, driver="GPKG")
            print(f"Ergebnis für {city_key} (2019-2024) gespeichert: {out_path}")

            # Speicher freigeben
            del result_gdf
            del gdf # Speicher für das GeoDataFrame freigeben
            del w # Speicher für die Gewichtsmatrix freigeben


        except Exception as e:
            print(f"Fehler bei der Bearbeitung von {city_key} (2019-2024): {e}")
            if 'gdf' in locals() and gdf is not None:
                 del gdf
            if 'w' in locals() and w is not None:
                 del w


    print("\nGi* Analyse für 2019-2024 abgeschlossen.")
    print("\nGesamte Gi* Analyse abgeschlossen.")


--- Verarbeitung für Zeitraum 2019-2024 ---

Städte, die verarbeitet werden (2019-2024): ['2019_2024_Aschaffenburg', '2019_2024_Augsburg', '2019_2024_Bamberg', '2019_2024_Bayreuth', '2019_2024_Erlangen', '2019_2024_Fürth', '2019_2024_Ingolstadt', '2019_2024_Kempten_(Allgäu)', '2019_2024_Landshut', '2019_2024_Munich', '2019_2024_Nuremberg', '2019_2024_Regensburg', '2019_2024_Rosenheim', '2019_2024_Schweinfurt', '2019_2024_Würzburg', '2019_2024_Passau']

Bearbeite 2019_2024_Aschaffenburg (2019-2024)...
  Lade GeoPackage für 2019_2024_Aschaffenburg (2019-2024)...
  GeoPackage für 2019_2024_Aschaffenburg (2019-2024) erfolgreich geladen (602424 Features).
  Lade Gewichtsmatrix für Aschaffenburg...


/usr/local/lib/python3.12/dist-packages/libpysal/io/iohandlers/gal.py:185: UserWarning: The weights matrix is not fully connected: 
 There are 23380 disconnected components.
  w = W(neighbors, id_order=ids)


  Gewichtsmatrix für Aschaffenburg erfolgreich geladen (602424 Beobachtungen).
  Starte Gi* Analyse für 2019_2024_Aschaffenburg (2019-2024)...
Fehler bei der Bearbeitung von 2019_2024_Aschaffenburg (2019-2024): A worker process managed by the executor was unexpectedly terminated. This could be caused by a segmentation fault while calling the function or by an excessive memory usage causing the Operating System to kill the worker.

The exit codes of the workers are {SIGKILL(-9)}
Detailed tracebacks of the workers should have been printed to stderr in the executor process if faulthandler was not disabled.

Bearbeite 2019_2024_Augsburg (2019-2024)...
  Lade GeoPackage für 2019_2024_Augsburg (2019-2024)...
  GeoPackage für 2019_2024_Augsburg (2019-2024) erfolgreich geladen (1676056 Features).
  Lade Gewichtsmatrix für Augsburg...


/usr/local/lib/python3.12/dist-packages/libpysal/io/iohandlers/gal.py:185: UserWarning: The weights matrix is not fully connected: 
 There are 63858 disconnected components.
  w = W(neighbors, id_order=ids)


  Gewichtsmatrix für Augsburg erfolgreich geladen (1676056 Beobachtungen).
  Starte Gi* Analyse für 2019_2024_Augsburg (2019-2024)...


KeyboardInterrupt: 